# Aula 2 — Preparação de imagens para classificação

Nesta aula usaremos imagens de ressonância magnética do conjunto **Alzheimer MRI — 4 classes**. O objetivo deste notebook é adquirir, inspecionar e dividir os dados antes de construir uma rede neural convolucional.

> Este material é exclusivamente didático. Os modelos e resultados não devem ser usados para diagnóstico clínico.

## Da tabela para a imagem

Na aula anterior, cada amostra era um vetor com 10 números. Agora cada amostra é uma imagem, representada por um tensor com três dimensões:

```text
canais × altura × largura
```

As imagens originais são em tons de cinza, mas serão convertidas para três canais RGB porque os pesos pré-treinados da ResNet esperam três canais. Os três canais conterão a mesma informação visual.

## Bibliotecas e reprodutibilidade

In [ ]:
from pathlib import Path
import random
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Configuração

Para tornar a aula executável em máquinas mais modestas, limitamos cada classe a no máximo 400 imagens. Esse limite reduz o tempo de treino, mas não corrige sozinho o desbalanceamento: a classe `ModerateDemented` possui apenas 64 imagens.

In [ ]:
def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Raiz do projeto não encontrada.")

PROJECT_ROOT = find_project_root()
KAGGLE_DATASET = "marcopinamonti/alzheimer-mri-4-classes-dataset"
DATA_DIR = PROJECT_ROOT / "data"
DATASET_DIR = DATA_DIR / "Alzheimer_MRI_4_classes_dataset"
ARCHIVE_PATH = DATA_DIR / "alzheimer-mri-4-classes-dataset.zip"
MANIFEST_PATH = DATA_DIR / "alzheimer_mri_manifest.csv"

CLASS_NAMES = [
    "NonDemented",
    "VeryMildDemented",
    "MildDemented",
    "ModerateDemented",
]
MAX_IMAGES_PER_CLASS = 400
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

DATA_DIR.mkdir(parents=True, exist_ok=True)

## Aquisição dos dados

O código segue três alternativas: reutiliza uma pasta já preparada, extrai um arquivo ZIP local ou baixa o dataset público por meio do `kagglehub`. O download pode exigir autenticação do Kaggle dependendo da configuração da conta.

In [ ]:
def is_dataset_root(path):
    path = Path(path)
    return path.is_dir() and all((path / name).is_dir() for name in CLASS_NAMES)


def find_dataset_root(base_path):
    base_path = Path(base_path)
    if is_dataset_root(base_path):
        return base_path
    for candidate in base_path.rglob("Alzheimer_MRI_4_classes_dataset"):
        if is_dataset_root(candidate):
            return candidate
    return None


if not is_dataset_root(DATASET_DIR):
    if ARCHIVE_PATH.exists():
        print(f"Extraindo {ARCHIVE_PATH}...")
        with zipfile.ZipFile(ARCHIVE_PATH) as archive:
            archive.extractall(DATA_DIR)
    else:
        import kagglehub

        print(f"Baixando {KAGGLE_DATASET}...")
        download_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        downloaded_root = find_dataset_root(download_dir)
        if downloaded_root is None:
            raise FileNotFoundError("As quatro pastas de classes não foram encontradas no download.")
        shutil.copytree(downloaded_root, DATASET_DIR, dirs_exist_ok=True)

dataset_root = find_dataset_root(DATA_DIR)
if dataset_root is None:
    raise FileNotFoundError("Dataset não encontrado após a preparação.")

print("Dataset encontrado em:", dataset_root.resolve())

## Construção do manifesto

Em vez de carregar todas as imagens na memória, salvaremos somente caminhos, classes e dimensões. O `Dataset` da próxima aula abrirá cada arquivo apenas quando o `DataLoader` solicitar uma amostra. Essa estratégia é chamada de carregamento **lazy**.

In [ ]:
rng = np.random.default_rng(SEED)
records = []

for class_index, class_name in enumerate(CLASS_NAMES):
    class_dir = dataset_root / class_name
    image_paths = sorted(
        path for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    sample_size = min(MAX_IMAGES_PER_CLASS, len(image_paths))
    selected_indices = rng.choice(len(image_paths), size=sample_size, replace=False)

    for index in selected_indices:
        image_path = image_paths[int(index)]
        with Image.open(image_path) as image:
            width, height = image.size
        records.append({
            "relative_path": image_path.relative_to(DATA_DIR).as_posix(),
            "class_name": class_name,
            "label": class_index,
            "width": width,
            "height": height,
        })

manifest = pd.DataFrame(records)
print(f"Imagens selecionadas: {len(manifest)}")
manifest.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=manifest, x="class_name", hue="class_name", ax=axes[0], legend=False)
axes[0].set_title("Imagens selecionadas por classe")
axes[0].tick_params(axis="x", rotation=25)

dimension_counts = manifest.value_counts(["width", "height"]).reset_index(name="count")
sns.barplot(data=dimension_counts, x="width", y="count", hue="height", ax=axes[1])
axes[1].set_title("Dimensões originais")
plt.tight_layout()
plt.show()

manifest.groupby("class_name").size().rename("imagens")

## Divisão em treino, validação e teste

Usaremos 70% para treino, 15% para validação e 15% para teste. A estratificação preserva aproximadamente a proporção das classes em cada conjunto. O teste permanecerá intocado até a avaliação final.

In [ ]:
train_val, test_manifest = train_test_split(
    manifest, test_size=0.15, random_state=SEED, stratify=manifest["label"]
)
train_manifest, val_manifest = train_test_split(
    train_val, test_size=0.15 / 0.85, random_state=SEED,
    stratify=train_val["label"]
)

train_manifest = train_manifest.assign(split="train")
val_manifest = val_manifest.assign(split="validation")
test_manifest = test_manifest.assign(split="test")

manifest = pd.concat([train_manifest, val_manifest, test_manifest], ignore_index=True)
manifest = manifest.sample(frac=1, random_state=SEED).reset_index(drop=True)
manifest.to_csv(MANIFEST_PATH, index=False)

split_table = pd.crosstab(manifest["split"], manifest["class_name"])
print(f"Manifesto salvo em: {MANIFEST_PATH.resolve()}")
split_table

## Visualização das amostras de treino

Inspecionar os dados visualmente ajuda a encontrar arquivos corrompidos, orientações inesperadas, artefatos e diferenças de contraste antes do treinamento.

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(9, 10))

for row, class_name in enumerate(CLASS_NAMES):
    examples = manifest.query("split == 'train' and class_name == @class_name").sample(
        n=3, random_state=SEED
    )
    for column, record in enumerate(examples.itertuples()):
        image = Image.open(DATA_DIR / record.relative_path).convert("L")
        axes[row, column].imshow(image, cmap="gray")
        axes[row, column].set_title(class_name)
        axes[row, column].axis("off")

plt.tight_layout()
plt.show()

## Limitações dos dados

- O conjunto é fortemente desbalanceado, principalmente na classe `ModerateDemented`.
- Os nomes dos arquivos não fornecem identificadores confiáveis de pacientes. Portanto, garantimos separação por **imagem**, não por paciente. Se existirem múltiplos cortes do mesmo paciente, pode haver vazamento entre os conjuntos.
- As classes representam estágios ordenados da doença, mas esta aula os tratará como quatro classes categóricas.
- O dataset é adequado para demonstração, não para validação clínica.

O próximo notebook usará o manifesto salvo para treinar uma CNN simples e uma ResNet-18.